# Lasso: Seleccion Automatica de Features
## Laboratorio guiado para ML1 (Tema 11 -- complemento)

**Universidad Externado de Colombia**
**Curso:** Machine Learning I (ML1-2026I)
**Docente:** Julian Zuluaga
**Duracion estimada:** 90 minutos en clase
**Pre-requisito:** Haber completado `nb-14-guia-regularizacion-diabetes.ipynb`

---

## Objetivo

En el laboratorio anterior viste **Ridge** y **Lasso** lado a lado sobre Diabetes.
Hoy nos enfocamos **solo en Lasso** pero desde un angulo distinto:
**Lasso como herramienta de seleccion de variables en un problema de negocio real.**

## Aprendizajes esperados

Al final del laboratorio deberias poder responder:

1. Por que Lasso es especialmente util cuando tienes **muchas features categoricas dummificadas**?
2. Como interpretas un modelo Lasso a un stakeholder no tecnico?
3. Que pasa con la seleccion de Lasso cuando hay **features correlacionadas**?
4. Como cambia el alpha optimo si reduces el tamanyo del train set?
5. Por que el numero de features activas es **una metrica de negocio**, no solo estadistica?

## Contexto de negocio

Trabajas como cientifico de datos en **una startup de bicicletas compartidas** (estilo Tembici, Lime, Bicirun).
Operations quiere predecir cuantas bicicletas se alquilaran cada hora para redistribuir la flota.

Tienes **dos anyos de datos historicos** de Capital Bikeshare (Washington DC) con muchas
variables candidatas: clima, hora, dia de la semana, festivo, estacion del anyo, temperatura
real vs. sensacion termica, humedad, viento.

**La pregunta de Operations:** "Dame las 10-15 senyales clave para construir un dashboard simple.
No quiero un modelo de 200 variables que no podamos explicar al equipo de campo."

**Esa pregunta es Lasso en una frase.**

## Como usar este notebook

1. Lee la celda de explicacion **antes** de tocar el codigo.
2. Anticipa el resultado **antes** de correr.
3. Completa los `TODO` siguiendo los hints.
4. Responde las **preguntas de discusion** en celdas markdown debajo.
5. No saltes secciones - el orden importa pedagogicamente.


---

## 0. Setup del entorno

### En Google Colab

Todas las librerias estan instaladas. No tienes que hacer nada.

### Localmente

```bash
pip install scikit-learn pandas numpy matplotlib seaborn
```

### Sobre el dataset

**Bike Sharing Demand** (Fanaee-T & Gama, 2013).
Origen: Capital Bikeshare, Washington DC, 2011-2012.

**Caracteristicas:**
- 17,379 registros (uno por hora durante 2 anyos)
- Target: `count` = numero total de bicicletas alquiladas en esa hora
- Source: UCI ML Repository / OpenML
- Documentacion: https://archive.ics.uci.edu/ml/datasets/bike+sharing+dataset

**Variables disponibles:**

| Variable | Tipo | Significado |
|----------|------|-------------|
| `season` | categorica (1-4) | invierno, primavera, verano, otonyo |
| `holiday` | binaria | dia festivo? |
| `workingday` | binaria | dia laboral (ni weekend ni festivo)? |
| `weather` | categorica (1-4) | desde despejado hasta tormenta |
| `temp` | continua | temperatura real (normalizada 0-1) |
| `atemp` | continua | sensacion termica (normalizada 0-1) |
| `humidity` | continua | humedad relativa (normalizada 0-1) |
| `windspeed` | continua | velocidad del viento (normalizada 0-1) |
| `casual` | continua | alquileres de usuarios NO registrados |
| `registered` | continua | alquileres de usuarios registrados |
| `count` | continua | **TARGET**: total alquileres = casual + registered |


In [ ]:
# Imports - corre esta celda primero (esta completa, no debes modificarla)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Lasso, LassoCV, Ridge, RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

# Reproducibilidad - todos obtienen los mismos resultados
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Estetica de graficos
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

print('Entorno listo')


---

## 1. Carga y exploracion del dataset

### Tarea 1.1 - Cargar el dataset desde OpenML

Usa `fetch_openml` para traer **Bike_Sharing_Demand**.

> **Hint:**
> ```python
> bike = fetch_openml(name='Bike_Sharing_Demand', version=2, as_frame=True)
> df = bike.frame
> ```
>
> Esto retorna un DataFrame completo. La primera descarga puede tardar 30-60 seg
> (se cachea localmente; las siguientes corridas son instantaneas).

**Verifica que cargaste bien:**
- `df.shape` deberia dar `(17379, 12)`
- `df.columns` deberia incluir `season`, `weather`, `temp`, `humidity`, `count`, etc.


In [ ]:
# TODO 1.1: cargar el dataset y guardarlo en df

bike = fetch_openml(name='Bike_Sharing_Demand', version=2, as_frame=True)
df = bike.frame

# Verificacion
print(f"Forma: {df.shape}")
print(f"Columnas: {list(df.columns)}")
df.head()

### Tarea 1.2 - Hipotesis ANTES de modelar

Antes de ver una sola correlacion o coeficiente, escribe en la celda markdown debajo
**tu intuicion del problema**. Esta seccion es **critica** -- al final del lab vas a
comparar tu intuicion con lo que Lasso seleccionara automaticamente.

Responde:

1. **Top-3 features que esperas que sean MAS importantes** para predecir `count`. Por que?
2. **Top-3 features que esperas que sean MENOS importantes** (o ruido). Por que?
3. **Una interaccion que crees que importa.** Ejemplo: "hora x dia_laboral" porque
   la demanda en hora pico solo aplica entre semana. Cual propones tu?

> **No googlees ni mires correlaciones.** Esto es pura intuicion de negocio.
> El objetivo es verificar mas adelante si Lasso confirma o refuta tu modelo mental.


**Tu hipotesis**:

**Features TOP-3 que predigo MAS importantes:**
1. `hour`: Vital para entender los ciclos de transporte diario (picos mañana/tarde).
2. `temp`: El clima es el principal factor externo para elegir bicicleta.
3. `workingday`: Define si el patrón es pendular (trabajo) o recreativo.

**Features que predigo MENOS importantes / ruido:**
1. `holiday`: Representa muy pocos registros del dataset.
2. `windspeed`: Salvo en tormentas, no suele disuadir al ciclista promedio.
3. `weekday`: Mucha de su varianza ya está en `workingday`.

**Una interaccion que creo que importa:**
`hour x workingday`: La demanda a las 8 AM es altísima entre semana, pero baja un domingo.

### Tarea 1.3 - EDA basico

Antes de modelar, mira los datos. Haz **tres** cosas:

1. `df.describe()` -- ojo a las escalas de cada variable.
2. Histograma del target: `plt.hist(df['count'], bins=50)`. Es simetrico? Sesgado?
3. `df.isna().sum()` -- hay nulos?


In [ ]:
# TODO 1.3: exploracion inicial
print(df.describe())
plt.hist(df['count'], bins=50)
plt.title('Distribución de Alquileres')
plt.show()
print(df.isna().sum())

---

## 2. Trampa de data leakage: las columnas que NO puedes usar

Mira las primeras filas:

```
casual + registered = count  ALWAYS
```

Esas dos columnas son **descomposiciones del target**. Si las dejas como features,
tu modelo va a aprender `count = casual + registered + 0` -- una identidad trivial
con R^2 = 1.0 y cero valor predictivo real.

**Esto se llama data leakage.** Es uno de los errores mas comunes de cientificos de datos juniors
y la razon mas frecuente de modelos que se "rompen" en produccion.

### Tarea 2.1 - Eliminar columnas con leakage

Elimina `casual` y `registered` del DataFrame. Tambien elimina cualquier columna de
fecha/timestamp si la hay (`datetime`, `dteday`).

> **Hints:**
> - `df = df.drop(columns=['casual', 'registered'])` (o con `inplace=True`).
> - Si hay una columna de fecha, eliminala tambien: las features temporales
>   importantes (hora, dia, mes) ya estan codificadas en columnas aparte.
> - Verifica al final con `df.columns`.


In [ ]:
# TODO 2.1: eliminar columnas con leakage
cols_leakage = [c for c in ['casual', 'registered', 'datetime', 'dteday'] if c in df.columns]
df = df.drop(columns=cols_leakage)
print(f"Columnas restantes: {list(df.columns)}")

### Tarea 2.2 - Separar X e y

- `y = df['count']` (target, lo que queremos predecir)
- `X = df.drop(columns=['count'])` (el resto son features candidatas)


In [ ]:
# TODO 2.2: separar features y target
y = df['count']
X = df.drop(columns=['count'])
print(f"X shape: {X.shape} | y shape: {y.shape}")

---

## 3. Preparacion de datos: encoding y muestreo

### Tarea 3.1 - One-hot encoding de variables categoricas

Las columnas `season` y `weather` son **categoricas**, pero estan codificadas como enteros
(1, 2, 3, 4). Si las dejas asi, el modelo lineal asumira que "season=4" es 4x mas grande
que "season=1", lo cual no tiene sentido (el invierno no es "4x verano").

**Solucion:** convertirlas a one-hot (dummies).

> **Hint:**
> ```python
> X = pd.get_dummies(X, columns=['season', 'weather'], drop_first=True, dtype=float)
> ```
>
> - `drop_first=True` evita la trampa de **dummy variable**: con K categorias generas K-1 columnas.
> - `dtype=float` asegura que las dummies queden como numericas (no `bool`).


In [ ]:
# TODO 3.1: one-hot encoding
X = pd.get_dummies(X, columns=['season', 'weather', 'holiday', 'workingday'], drop_first=True, dtype=float)
print(f"Features tras encoding: {X.shape[1]}")

### Tarea 3.2 - Subsampling: simular escenario "startup con pocos datos"

**Por que subsamplear?**

Bike Sharing tiene 17,379 filas. Con tantos datos, **OLS NO sobreajusta** ni siquiera con
muchas features -- la regla `n >> p` lo salva.

Pero la realidad de un cientifico de datos en una startup es distinta: pocos datos historicos,
poco presupuesto, muchas features candidatas que el equipo de producto quiere probar.

Vamos a simular ese escenario: **muestrear solo 1,000 filas**.

> **Hint:**
> ```python
> sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X), size=1000, replace=False)
> X = X.iloc[sample_idx].reset_index(drop=True)
> y = y.iloc[sample_idx].reset_index(drop=True)
> ```


In [ ]:
# TODO 3.2: subsample a 1000 filas
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X), size=1000, replace=False)
X = X.iloc[sample_idx].reset_index(drop=True)
y = y.iloc[sample_idx].reset_index(drop=True)
print(f"X shape post-sample: {X.shape}")

### Tarea 3.3 - Split train/test (80/20)

- `train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)`
- Espera: 800 train, 200 test.

> **Regla de oro recordatoria:** el test set **NO se toca** hasta el final.
> Nada de tunear alpha mirando metricas en test.


In [ ]:
# TODO 3.3: split 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

---

## 4. Linea base: OLS con features crudas

Antes de meternos en problemas, veamos como se comporta una regresion lineal "honesta"
con las features tal como vienen (sin interacciones polinomicas).

### Tarea 4.1 - Entrena OLS y reporta metricas

- Modelo: `LinearRegression()`
- Entrenamiento: `X_train, y_train`
- Evaluacion: Train R^2, Test R^2, Gap


In [ ]:
# TODO 4.1: OLS sobre features crudas
ols_base = LinearRegression().fit(X_train, y_train)
train_r2_base = r2_score(y_train, ols_base.predict(X_train))
test_r2_base = r2_score(y_test, ols_base.predict(X_test))
gap_base = train_r2_base - test_r2_base
print(f"OLS base | Train R2: {train_r2_base:.3f} | Test R2: {test_r2_base:.3f} | Gap: {gap_base:.3f}")

**Discusion 4.1**:

1. **Test R^2**: ~0.256. Es bajo. Operations diría que el modelo solo explica el 25% de la varianza, lo cual es insuficiente para decisiones críticas.
2. **Sobreajuste**: El gap es de ~0.16. Hay un sobreajuste moderado, lo cual es raro con p=15, sugiriendo que la muestra de 800 filas aún es ruidosa para un modelo lineal simple.
3. **Coincidencia**: No coincide totalmente. Se esperaba menos overfit. Quizás la naturaleza de los datos (altamente no lineales como 'hour') confunde al modelo lineal.

---

## 5. Forzando el sobreajuste: PolynomialFeatures(degree=2)

"Y si las interacciones importan? Y si `temp x humidity` o `windspeed^2` predicen mejor?"

Vamos a generar **todas las interacciones de segundo orden** entre las features.

### Tarea 5.1 - Expandir features con PolynomialFeatures

Usa `PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)`.

> **CUIDADO con data leakage del transformer:**
> - `poly.fit_transform(X_train)` <- aqui fit + transform sobre TRAIN
> - `poly.transform(X_test)` <- aqui SOLO transform sobre TEST
> - **Nunca** `fit` sobre la union train+test.

> **Hints:**
> ```python
> poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
> X_train_poly = poly.fit_transform(X_train)
> X_test_poly  = poly.transform(X_test)
> ```


In [ ]:
# TODO 5.1: expansion polinomica grado 2
poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
print(f"Features originales: {X_train.shape[1]} | Tras expansion: {X_train_poly.shape[1]}")

### Tarea 5.2 - OLS sobre features expandidas

Ajusta `LinearRegression` sobre `X_train_poly` y evalua igual que antes.

> **Prediccion del docente:** Train R^2 va a subir, Test R^2 va a caer (puede ser negativo).
> Esto es **sobreajuste en vivo**: el modelo memorizo el ruido del train.


In [ ]:
# TODO 5.2: OLS con features expandidas
ols_poly = LinearRegression().fit(X_train_poly, y_train)
train_r2_poly = r2_score(y_train, ols_poly.predict(X_train_poly))
test_r2_poly = r2_score(y_test, ols_poly.predict(X_test_poly))
gap_poly = train_r2_poly - test_r2_poly
print(f"OLS poly | Train R2: {train_r2_poly:.3f} | Test R2: {test_r2_poly:.3f} | Gap: {gap_poly:.3f}")

**Discusion 5.2**:

1. **Cambio**: El Train R2 subió a 0.61, pero el Test R2 es ~0.35. El gap aumentó significativamente. El modelo está memorizando el ruido.
2. **Coeficientes**: No explotaron a niveles absurdos en esta corrida, pero suelen ser inestables. Un coeficiente gigante no tiene sentido físico.
3. **Negocio**: Operations diría que el modelo es una 'caja negra' con 150+ variables imposible de explicar al equipo de campo.

---

## 6. Lasso al rescate: penalizacion L1

**Recordatorio rapido de teoria** (vista en `nb-14`):

$$
\mathcal{L}_{Lasso}(\beta) = \sum_i (y_i - \hat{y}_i)^2 + \alpha \sum_j |\beta_j|
$$

- Primer termino: error cuadratico (igual que OLS).
- Segundo termino: penalizacion por **valor absoluto** de cada coeficiente.
- `alpha = 0` -> equivale a OLS.
- `alpha -> infinito` -> todos los coeficientes en cero.

**Diferencia clave con Ridge:** Lasso **anula** coeficientes (los lleva exactamente a cero).
No los encoge -- los **descarta**. Esto lo convierte en un **selector de variables automatico**.

### Tarea 6.1 - Pipeline OBLIGATORIO: StandardScaler + Lasso

> **CRITICO:** antes de aplicar Lasso, **estandariza las features**.
>
> Por que? La penalizacion `alpha * sum(|beta_j|)` trata cada coeficiente por igual.
> Pero si una feature tiene escala 0-1 (como `humidity`) y otra escala 0-100 (si la hubiera),
> sus coeficientes naturales viven en universos distintos. La penalizacion no es justa
> sin estandarizar.

Construye un Pipeline con dos pasos:

1. `("scaler", StandardScaler())`
2. `("lasso", Lasso(alpha=1.0, max_iter=10000))`

> **Por que `max_iter=10000`?** Lasso resuelve un problema de optimizacion convexa pero NO
> diferenciable (por el valor absoluto). El solver iterativo puede necesitar muchas vueltas
> para converger, especialmente con muchas features. Si ves un `ConvergenceWarning`,
> sube max_iter.

Entrena con `X_train_poly`, evalua en train y test.


In [ ]:
# TODO 6.1: Pipeline Scaler + Lasso(alpha=1.0)
pipe_lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(alpha=1.0, max_iter=10000))
])
pipe_lasso.fit(X_train_poly, y_train)
train_r2_lasso_1 = r2_score(y_train, pipe_lasso.predict(X_train_poly))
test_r2_lasso_1 = r2_score(y_test, pipe_lasso.predict(X_test_poly))
coefs_1 = pipe_lasso.named_steps['lasso'].coef_
n_activas_1 = np.sum(coefs_1 != 0)
print(f"Lasso(1.0) | Train R2: {train_r2_lasso_1:.3f} | Test R2: {test_r2_lasso_1:.3f} | Activas: {n_activas_1}")

**Discusion 6.1**:

1. Con `alpha=1.0`, cuantas features sobrevivieron (coef != 0)?
2. Es mejor que OLS poly en Test R^2? Por que?
3. Pero `alpha=1.0` fue una **eleccion arbitraria**. Por que es trampa elegir alpha "a ojo"?

---

**Tu respuesta:**

> _(double-click para editar)_


---

## 7. Regularization Path: ver desaparecer las features

Vamos a graficar como cambian los coeficientes a medida que `alpha` crece.
En Ridge, el path era una **convergencia suave** hacia cero. En Lasso, vas a ver algo distinto:
los coeficientes **caen de golpe** a cero, uno a uno, en orden de importancia.

### Tarea 7.1 - Calcular y graficar el path

**Pasos:**

1. Define una grilla logaritmica de alphas: `alphas = np.logspace(-3, 1, 50)`.
2. Para cada alpha, entrena un pipeline `Scaler + Lasso(alpha)` y guarda los coeficientes.
3. Apila todo en un array `(50, n_features)`.
4. Grafica:
   - Eje X: alpha en escala log
   - Eje Y: valor de cada coeficiente
   - Una linea por feature

> **Lo que deberias ver:**
> - Con alpha muy bajo: muchas features activas, valores dispersos
> - Al subir alpha: features se van **anulando una a una**
> - Con alpha alto: todas en cero

> **Hints:**
> ```python
> alphas = np.logspace(-3, 1, 50)
> coefs_path = []
> for a in alphas:
>     pipe = Pipeline([("scaler", StandardScaler()), ("lasso", Lasso(alpha=a, max_iter=10000))])
>     pipe.fit(X_train_poly, y_train)
>     coefs_path.append(pipe.named_steps['lasso'].coef_)
> coefs_path = np.array(coefs_path)
>
> plt.figure(figsize=(10, 6))
> plt.plot(alphas, coefs_path)
> plt.xscale('log')
> plt.xlabel('alpha (log)')
> plt.ylabel('Coeficientes')
> plt.title('Lasso Regularization Path')
> plt.show()
> ```


In [ ]:
# TODO 7.1: regularization path de Lasso
alphas = np.logspace(-3, 1, 50)
coefs_path = []
for a in alphas:
    pipe = Pipeline([("scaler", StandardScaler()), ("lasso", Lasso(alpha=a, max_iter=10000))])
    pipe.fit(X_train_poly, y_train)
    coefs_path.append(pipe.named_steps['lasso'].coef_)
coefs_path = np.array(coefs_path)
plt.figure(figsize=(10, 6))
plt.plot(alphas, coefs_path)
plt.xscale('log')
plt.xlabel('alpha (log)')
plt.ylabel('Coeficientes')
plt.title('Lasso Regularization Path')
plt.show()

### Tarea 7.2 - Curva de features activas vs alpha

Complementa el path con un grafico adicional:

- Eje X: alpha (log)
- Eje Y: numero de features con coef != 0

> **Esto es una vista de "compresion": cuantas variables sobreviven a cada nivel de penalizacion?**

> **Hint:**
> ```python
> n_activas_path = (coefs_path != 0).sum(axis=1)
> plt.plot(alphas, n_activas_path)
> plt.xscale('log')
> plt.xlabel('alpha (log)')
> plt.ylabel('Numero de features activas')
> plt.title('Sparsidad inducida por Lasso')
> ```


In [ ]:
# TODO 7.2: curva de features activas vs alpha
n_activas_path = (coefs_path != 0).sum(axis=1)
plt.plot(alphas, n_activas_path)
plt.xscale('log')
plt.xlabel('alpha (log)')
plt.ylabel('# features activas')
plt.title('Sparsidad inducida por Lasso')
plt.show()

---

## 8. LassoCV: elegir alpha sin trampa

Elegir alpha mirando el test set es **data leakage**. La forma correcta es
**validacion cruzada en el train set**: `LassoCV` prueba multiples alphas
internamente con CV y se queda con el mejor.

### Tarea 8.1 - Pipeline con LassoCV

> **Hints:**
> ```python
> pipe_lasso_cv = Pipeline([
>     ("scaler", StandardScaler()),
>     ("lasso_cv", LassoCV(alphas=np.logspace(-3, 1, 50), cv=5, max_iter=10000, random_state=RANDOM_STATE))
> ])
> pipe_lasso_cv.fit(X_train_poly, y_train)
> ```

Reporta:
- **alpha optimo:** `pipe_lasso_cv.named_steps['lasso_cv'].alpha_`
- **Train R^2, Test R^2, Gap**
- **Numero de features activas**


In [ ]:
# TODO 8.1: LassoCV con CV=5
pipe_lasso_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso_cv", LassoCV(alphas=np.logspace(-3, 1, 50), cv=5, max_iter=10000, random_state=RANDOM_STATE))
])
pipe_lasso_cv.fit(X_train_poly, y_train)
alpha_opt = pipe_lasso_cv.named_steps['lasso_cv'].alpha_
train_r2_cv = r2_score(y_train, pipe_lasso_cv.predict(X_train_poly))
test_r2_cv = r2_score(y_test, pipe_lasso_cv.predict(X_test_poly))
coefs_cv = pipe_lasso_cv.named_steps['lasso_cv'].coef_
n_activas_cv = np.sum(coefs_cv != 0)
print(f"Alpha optimo: {alpha_opt:.4f}")
print(f"LassoCV | Train R2: {train_r2_cv:.3f} | Test R2: {test_r2_cv:.3f} | Activas: {n_activas_cv}")

**Discusion 8.1**:

1. Cual fue el alpha optimo? Esta cerca de 1.0 (lo que usaste arbitrariamente en 6.1) o lejos?
2. Comparado con OLS poly (Seccion 5.2): cuanto subio o bajo el Test R^2?
3. **Compresion:** redujiste de `{p_poly}` features a `n_activas_cv` features. Que % es eso?
   Imagina llevar ese resultado a Operations: "elimine el X% del modelo sin sacrificar precision".

---

**Tu respuesta:**

> _(double-click para editar)_


---

## 9. Lo importante: que features sobrevivieron?

Aqui esta el **valor de negocio** de Lasso. Las features con coef != 0 son las que
el modelo considera importantes. Vamos a verlas **por nombre**.

### Tarea 9.1 - Obtener nombres de features polinomicas

Necesitas mapear los 200+ coeficientes a sus nombres originales.

> **Hints:**
> ```python
> poly_names = poly.get_feature_names_out(X_train.columns)
> # poly_names ahora es un array con nombres como:
> # 'temp', 'humidity', 'temp humidity', 'temp^2', 'workingday weather_3', etc.
> ```


In [ ]:
# TODO 9.1: obtener nombres y DataFrame de features
poly_names = poly.get_feature_names_out(X_train.columns)
coefs_lasso = pipe_lasso_cv.named_steps['lasso_cv'].coef_
mask = coefs_lasso != 0
df_seleccionadas = pd.DataFrame({
    'feature': poly_names[mask],
    'coef': coefs_lasso[mask],
})
df_seleccionadas['abs_coef'] = df_seleccionadas['coef'].abs()
df_seleccionadas = df_seleccionadas.sort_values('abs_coef', ascending=False).reset_index(drop=True)
print(f"Features seleccionadas: {len(df_seleccionadas)}")
print(df_seleccionadas.head(20))

### Tarea 9.2 - Visualizacion de las top 20 features

Grafica un bar plot horizontal con las **20 features mas importantes** (por magnitud absoluta de coeficiente).

> **Hint:**
> ```python
> top20 = df_seleccionadas.head(20).iloc[::-1]  # invertir para que la mas grande quede arriba
> plt.figure(figsize=(10, 8))
> plt.barh(top20['feature'], top20['coef'])
> plt.xlabel('Coeficiente (estandarizado)')
> plt.title('Top 20 features seleccionadas por Lasso')
> plt.tight_layout()
> ```


In [ ]:
# TODO 9.2: bar plot horizontal de top 20
top20 = df_seleccionadas.head(20).iloc[::-1]
plt.figure(figsize=(10, 8))
plt.barh(top20['feature'], top20['coef'])
plt.xlabel('Coeficiente (estandarizado)')
plt.title('Top 20 features seleccionadas por Lasso')
plt.tight_layout()
plt.show()

### Discusion 9 - El momento de la verdad

Vuelve a la **Tarea 1.2** donde anotaste tus hipotesis a priori.

Responde en celda markdown debajo:

1. **Tus top-3 features predichas vs. las top-3 de Lasso:** coinciden? cuales acerto y cuales no?
2. **Las features que predijiste como ruido:** Lasso las descarto (coef = 0)? O sobrevivieron?
3. **La interaccion que propusiste:** aparece en la lista de Lasso?
4. **Sorpresas:** hay alguna feature en el top que no esperabas? Como la explicarias clinicamente
   o "de negocio" a Operations?

> **Reflexion mas profunda:** un cientifico de datos sin Lasso habria tenido que decidir
> manualmente que features incluir. Tu intuicion estuvo cerca? La interpretabilidad **automatica**
> es el superpoder de Lasso.

---

**Tu respuesta:**

> _(double-click para editar)_


---

## 10. Tabla resumen comparativa

### Tarea 10.1 - Construye un DataFrame con todos los modelos entrenados

| Modelo                  | # Features | Train R^2 | Test R^2 | Gap | # Coefs != 0 |
|-------------------------|-----------|-----------|----------|-----|--------------|
| OLS (crudo)             | ?         | ?         | ?        | ?   | ?            |
| OLS (poly degree 2)     | ?         | ?         | ?        | ?   | ?            |
| Lasso (alpha=1.0)       | ?         | ?         | ?        | ?   | ?            |
| Lasso (CV optimo)       | ?         | ?         | ?        | ?   | ?            |

> **Hint:**
> ```python
> resumen = pd.DataFrame({
>     'modelo':        ['OLS crudo', 'OLS poly', 'Lasso(1.0)', 'Lasso CV'],
>     'n_features':    [X_train.shape[1], X_train_poly.shape[1], X_train_poly.shape[1], X_train_poly.shape[1]],
>     'train_r2':      [train_r2_base, train_r2_poly, train_r2_lasso_1, train_r2_cv],
>     'test_r2':       [test_r2_base,  test_r2_poly,  test_r2_lasso_1,  test_r2_cv],
>     'gap':           [...],
>     'coefs_no_cero': [X_train.shape[1], X_train_poly.shape[1], n_activas_1, n_activas_cv],
> })
> resumen
> ```


In [ ]:
# TODO 10.1: tabla resumen
resumen = pd.DataFrame({
    'modelo': ['OLS crudo', 'OLS poly', 'Lasso(1.0)', 'Lasso CV'],
    'n_features': [X_train.shape[1], X_train_poly.shape[1], X_train_poly.shape[1], X_train_poly.shape[1]],
    'train_r2': [train_r2_base, train_r2_poly, train_r2_lasso_1, train_r2_cv],
    'test_r2': [test_r2_base, test_r2_poly, test_r2_lasso_1, test_r2_cv],
    'coefs_no_cero': [X_train.shape[1], X_train_poly.shape[1], n_activas_1, n_activas_cv]
})
resumen['gap'] = resumen['train_r2'] - resumen['test_r2']
print(resumen)

**Pregunta 1 - Sobreajuste y diagnostico:**
OLS poly tiene un Train R2 de 0.610 vs un Test R2 de 0.353, con un gap de 0.257. Esta diferencia tan grande es evidencia clara de sobreajuste: el modelo funciona mucho mejor en datos conocidos que en nuevos.

---

## 12. Retos bonus (opcional, para casa)

### Reto 12.1 - Estabilidad bajo cambios de muestra

Repite todo el flujo con **5 random states distintos** en `train_test_split` y `subsample`.
Construye una tabla con:
- alpha optimo en cada run
- numero de features activas en cada run
- features que aparecen en TODAS las 5 corridas (las "estables")
- features que aparecen en solo 1-2 corridas (las "inestables")

> **Hint:** la tecnica formal se llama **Stability Selection** (Meinshausen & Buhlmann, 2010).

### Reto 12.2 - ElasticNet: lo mejor de los dos mundos

`ElasticNet` combina L1 y L2:

$$
\mathcal{L}_{EN}(\beta) = \sum_i (y_i - \hat{y}_i)^2 + \alpha \cdot \rho \cdot ||\beta||_1 + \frac{\alpha (1-\rho)}{2} \cdot ||\beta||_2^2
$$

Donde `rho = l1_ratio` controla la mezcla.

Usa `ElasticNetCV` con `l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9]`.

Mejora el Test R^2 vs Lasso puro? Cuantas features sobreviven? Es mas o menos sparso?

### Reto 12.3 - Reduccion de muestra: cuando Lasso brilla mas

Repite el ejercicio reduciendo el subsample a **200 filas** en vez de 1,000.

Observa: como cambia el alpha optimo? Por que? Que pasa con OLS poly en ese regimen?

> **Hint conceptual:** mientras menos datos, **mas conservador** es Lasso (alpha sube),
> porque con poca evidencia el modelo "desconfia" mas de cada feature.

### Reto 12.4 - Comparacion con seleccion clasica

Compara la seleccion de Lasso con dos metodos clasicos:
- `SelectKBest` con `f_regression` (filtro univariado)
- `RFE` con `LinearRegression` (eliminacion recursiva)

Pregunta: **comparten el top-10?** Cuantas features eligen los tres metodos en comun?
Cual da mejor Test R^2?

---

**Fin del laboratorio.**

Sube tu notebook completado al correo del docente antes del proximo encuentro.

**Recuerda:**
- Codigo limpio y comentado
- Todas las celdas markdown de discusion respondidas
- Graficos con titulos y ejes etiquetados
- **Tu intuicion inicial (Seccion 1.2) NO se borra** -- es parte del entregable
